In [37]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from math import sqrt
import random
random.seed(42)

In [38]:
data = pd.read_csv("../../data/ice_cream_sales.csv")
data.head()

,ID,Temperature,Is_Weekend,Hours_Open,Electricity_Usage,Ice_Cream_Sales
0,1,24.273285,False,10,92.118314,983
1,2,25.503474,False,9,83.917817,1018
2,3,24.370024,False,8,88.290617,951
3,4,24.377495,False,10,85.561865,1012
4,5,26.632614,False,8,94.404976,1010


In [39]:
# Multi variate regression

X = data.drop(["Ice_Cream_Sales", "ID", "Is_Weekend", "Electricity_Usage"], axis=1)
y_electricity = data["Electricity_Usage"]

X_train, X_test, y_train_electricity, y_test_electricity = train_test_split(X, y_electricity, test_size=0.2, random_state=42)

In [40]:
model_electricity = LinearRegression()
model_electricity.fit(X_train, y_train_electricity)
mv_data = data.copy()
mv_data['Predicted_Electricity_Usage'] = model_electricity.predict(X)

In [41]:
print(f"Intercept: {model_electricity.intercept_}")
for i in range(len(X.columns)):
    print(f'{X.columns[i]}: {model_electricity.coef_[i]}')

Intercept: 9.989685067743395
Temperature: 2.0330029490608394
Hours_Open: 2.9774139032260956


In [42]:
mse_electricity = mean_squared_error(y_electricity, mv_data['Predicted_Electricity_Usage'])
print(f'Mean Error: {sqrt(mse_electricity)}')

Mean Error: 7.061551865348178


In [43]:
X_ics = mv_data.drop(["Ice_Cream_Sales", "ID", "Is_Weekend", "Electricity_Usage"], axis=1)
y_ics = mv_data["Ice_Cream_Sales"]

X_train, X_test, y_train_ics, y_test_ics = train_test_split(X_ics, y_ics, test_size=0.2, random_state=42)

In [44]:
model_ice_cream = Ridge(alpha=1)
model_ice_cream.fit(X_train, y_train_ics)
y_pred_ice_cream = model_ice_cream.predict(X_test)

In [45]:
mse_ice_cream = mean_squared_error(y_test_ics, y_pred_ice_cream)
print(f'Mean Error for Ice Cream Sales: {sqrt(mse_ice_cream)}')

Mean Error for Ice Cream Sales: 39.93458057597932


In [46]:
print(f'Intercept: {model_ice_cream.intercept_}')
for i in range(len(X_ics.columns)):
    print(f'{X_ics.columns[i]}: {model_ice_cream.coef_[i]}')

Intercept: -380.3376183506673
Temperature: -3.1916996873849244
Hours_Open: 7.562240766532567
Predicted_Electricity_Usage: 16.02718592086218


In [47]:
# Intervention Analysis

data = pd.read_csv("../../data/ice_cream_sales.csv")
data["Electricity_Usage"] = data["Electricity_Usage"] + 20
data.rename(columns={"Electricity_Usage": "Predicted_Electricity_Usage"}, inplace=True)
X = data.drop(["Ice_Cream_Sales", "Is_Weekend", "ID"], axis=1)
y = data["Ice_Cream_Sales"];

In [48]:
data["Predicted_Ice_Cream_Sales"] = model_ice_cream.predict(X)
mse_ice_cream = mean_squared_error(y, data["Predicted_Ice_Cream_Sales"])
print(f'Mean Error for Ice Cream Sales: {sqrt(mse_ice_cream)}')

Mean Error for Ice Cream Sales: 338.9092707331405


In [49]:
# Counterfactual analysis

data = pd.read_csv("../../data/ice_cream_sales.csv")
data["Ice_Cream_Sales"] = data["Ice_Cream_Sales"] + (30 - data["Temperature"]) * 30
data["Temperature"] = 30
X = data.drop(["Ice_Cream_Sales", "Is_Weekend", "ID", "Electricity_Usage"], axis=1)
y_electricity = data["Electricity_Usage"];

In [50]:
data["Predicted_Electricity_Usage"] = model_electricity.predict(X)

In [51]:
X_ics = data.drop(["Ice_Cream_Sales", "Is_Weekend", "ID", "Electricity_Usage"], axis=1)
data["Predicted_Ice_Cream_Sales"] = model_ice_cream.predict(X_ics)

print(data[["Ice_Cream_Sales", "Predicted_Ice_Cream_Sales", "Predicted_Electricity_Usage"]].head())

   Ice_Cream_Sales  Predicted_Ice_Cream_Sales  Predicted_Electricity_Usage
0      1154.801449                1214.335488                   100.753913
1      1152.895767                1159.053681                    97.776499
2      1119.899278                1103.771874                    94.799085
3      1180.675144                1214.335488                   100.753913
4      1111.021594                1103.771874                    94.799085


In [52]:
mse_ics = mean_squared_error(data["Ice_Cream_Sales"], data["Predicted_Ice_Cream_Sales"])
print(f'Mean Error for Ice Cream Sales: {sqrt(mse_ics)}')

Mean Error for Ice Cream Sales: 38.02179083226328
